# PINNs for Data Center Cooling Optimisation

This notebook walks through the complete pipeline:
1. Generate a synthetic 2-D data center environment
2. Train a Physics-Informed Neural Network (PINN)
3. Visualise temperature predictions and hotspots
4. Run cooling-vent optimisation

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Data Generation

In [ ]:
from project.data.generate import (
    generate_steady_state_temperature,
    generate_time_dependent_data,
    prepare_training_data,
)

data = generate_steady_state_temperature()
train_data = prepare_training_data(data)

print('Grid shape:', data['T'].shape)
print('Temp range:', data['T'].min(), '-', data['T'].max())

In [ ]:
from project.visualization.plots import plot_temperature_heatmap

fig = plot_temperature_heatmap(
    data['X'], data['Y'], data['T'],
    title='Ground-Truth Temperature',
    server_racks=data['config']['server_racks'],
    cooling_vents=data['config']['cooling_vents'],
)
plt.show()

## 2. PINN Training

In [ ]:
from project.pinn.train import train

train_data['config'] = data['config']
result = train(
    train_data,
    layers=[2, 64, 64, 64, 1],
    epochs=2000,
    log_every=500,
)
model = result['model']
history = result['history']

In [ ]:
from project.visualization.plots import plot_training_loss

fig = plot_training_loss(history)
plt.show()

## 3. PINN Evaluation

In [ ]:
X, Y = data['X'], data['Y']
xy_grid = np.stack([X.ravel(), Y.ravel()], axis=1)

model.eval()
with torch.no_grad():
    T_pred = model(
        torch.tensor(xy_grid, dtype=torch.float32)
    ).numpy().reshape(X.shape)

fig = plot_temperature_heatmap(
    X, Y, T_pred,
    title='PINN Prediction',
    server_racks=data['config']['server_racks'],
    cooling_vents=data['config']['cooling_vents'],
)
plt.show()

In [ ]:
from project.visualization.plots import plot_hotspot_detection, plot_airflow

fig = plot_hotspot_detection(X, Y, T_pred)
plt.show()

fig = plot_airflow(X, Y, T_pred)
plt.show()

## 4. Cooling Optimisation

In [ ]:
from project.optimization.cooling_optimizer import optimize_vent_placement, suggest_improvements
from project.visualization.plots import plot_optimization_comparison

opt = optimize_vent_placement(n_candidates=40, seed=42)
for s in suggest_improvements(opt):
    print(s)

fig = plot_optimization_comparison(
    X, Y, data['T'], opt['best_T']
)
plt.show()